# Agent Comparison

This notebook compares the current Hanabi baselines:

- `RandomAgent`
- `BasicHeuristicAgent`
- `ConservativeHeuristicAgent`

It is meant to serve as the first lightweight record of agent metrics in the repository.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\marce\Documents\GitHub\hanabi-ai


In [2]:
from card_game_ai.agents.random_agent import RandomAgent
from card_game_ai.agents.heuristic.basic_agent import BasicHeuristicAgent
from card_game_ai.agents.heuristic.conservative_agent import ConservativeHeuristicAgent
from card_game_ai.training.self_play import evaluate_self_play


In [3]:
# Edit these values when you want to record a new comparison run.
PLAYER_COUNT = 2
GAME_COUNT = 200
SEED_BASE = 0
AGENT_SEED_BASE = 1000

In [4]:
basic_evaluation = evaluate_self_play(
    lambda player_id, game_index: BasicHeuristicAgent(),
    player_count=PLAYER_COUNT,
    game_count=GAME_COUNT,
    seed_base=SEED_BASE,
)

conservative_evaluation = evaluate_self_play(
    lambda player_id, game_index: ConservativeHeuristicAgent(),
    player_count=PLAYER_COUNT,
    game_count=GAME_COUNT,
    seed_base=SEED_BASE,
)

random_evaluation = evaluate_self_play(
    lambda player_id, game_index: RandomAgent(
        seed=AGENT_SEED_BASE + (game_index * PLAYER_COUNT) + player_id
    ),
    player_count=PLAYER_COUNT,
    game_count=GAME_COUNT,
    seed_base=SEED_BASE,
)


In [ ]:
def evaluation_to_row(name, evaluation):
    return {
        "agent": name,
        "games": evaluation.game_count,
        "players": evaluation.player_count,
        "avg_score": round(evaluation.average_score, 3),
        "median_score": round(evaluation.median_score, 3),
        "min_score": evaluation.min_score,
        "max_score": evaluation.max_score,
        "avg_turns": round(evaluation.average_turn_count, 3),
        "avg_hints": round(evaluation.average_hint_tokens, 3),
        "avg_strikes": round(evaluation.average_strike_tokens, 3),
        "avg_completed_stacks": round(evaluation.average_completed_stacks, 3),
        "win_rate": round(evaluation.win_rate, 4),
        "loss_rate": round(evaluation.loss_rate, 4),
        "score>=10": round(evaluation.score_at_least_10_rate, 4),
        "score>=15": round(evaluation.score_at_least_15_rate, 4),
    }


rows = [
    evaluation_to_row("RandomAgent", random_evaluation),
    evaluation_to_row("BasicHeuristicAgent", basic_evaluation),
    evaluation_to_row("ConservativeHeuristicAgent", conservative_evaluation),
]

In [8]:
rows[0]

{'agent': 'RandomAgent',
 'games': 200,
 'players': 2,
 'avg_score': 1.415,
 'median_score': 1.0,
 'min_score': 0,
 'max_score': 8,
 'avg_turns': 13.93,
 'avg_hints': 4.855,
 'avg_strikes': 3.0,
 'avg_completed_stacks': 0.0,
 'win_rate': 0.0,
 'loss_rate': 1.0,
 'score>=10': 0.0,
 'score>=15': 0.0}

In [9]:
rows[1]

{'agent': 'BasicHeuristicAgent',
 'games': 200,
 'players': 2,
 'avg_score': 5.54,
 'median_score': 6.0,
 'min_score': 1,
 'max_score': 12,
 'avg_turns': 82.68,
 'avg_hints': 0.185,
 'avg_strikes': 0.0,
 'avg_completed_stacks': 0.005,
 'win_rate': 0.0,
 'loss_rate': 0.0,
 'score>=10': 0.035,
 'score>=15': 0.0}

In [10]:
rows[2]

{'agent': 'ConservativeHeuristicAgent',
 'games': 200,
 'players': 2,
 'avg_score': 8.25,
 'median_score': 8.0,
 'min_score': 3,
 'max_score': 16,
 'avg_turns': 80.155,
 'avg_hints': 0.305,
 'avg_strikes': 0.0,
 'avg_completed_stacks': 0.07,
 'win_rate': 0.0,
 'loss_rate': 0.0,
 'score>=10': 0.33,
 'score>=15': 0.02}

In [6]:
headers = list(rows[0].keys())
widths = {
    header: max(len(header), max(len(str(row[header])) for row in rows))
    for header in headers
}

header_line = " | ".join(header.ljust(widths[header]) for header in headers)
separator_line = "-+-".join("-" * widths[header] for header in headers)

print(header_line)
print(separator_line)
for row in rows:
    print(" | ".join(str(row[header]).ljust(widths[header]) for header in headers))

agent                      | games | players | avg_score | median_score | min_score | max_score | avg_turns | avg_hints | avg_strikes | avg_completed_stacks | win_rate | loss_rate | score>=10 | score>=15
---------------------------+-------+---------+-----------+--------------+-----------+-----------+-----------+-----------+-------------+----------------------+----------+-----------+-----------+----------
RandomAgent                | 200   | 2       | 1.415     | 1.0          | 0         | 8         | 13.93     | 4.855     | 3.0         | 0.0                  | 0.0      | 1.0       | 0.0       | 0.0      
BasicHeuristicAgent        | 200   | 2       | 5.54      | 6.0          | 1         | 12        | 82.68     | 0.185     | 0.0         | 0.005                | 0.0      | 0.0       | 0.035     | 0.0      
ConservativeHeuristicAgent | 200   | 2       | 8.25      | 8.0          | 3         | 16        | 80.155    | 0.305     | 0.0         | 0.07                 | 0.0      | 0.0       | 0.

In [ ]:
for name, evaluation in [
    ("RandomAgent", random_evaluation),
    ("BasicHeuristicAgent", basic_evaluation),
    ("ConservativeHeuristicAgent", conservative_evaluation),
]:
    print(name)
    # score_distribution is a tuple of (score, count) pairs.
    print(evaluation.score_distribution)
    print()

RandomAgent
((0, 57), (1, 66), (2, 39), (3, 25), (4, 6), (5, 4), (6, 2), (8, 1))

BasicHeuristicAgent
((1, 6), (2, 16), (3, 23), (4, 23), (5, 31), (6, 33), (7, 24), (8, 18), (9, 19), (10, 5), (11, 1), (12, 1))

ConservativeHeuristicAgent
((3, 2), (4, 12), (5, 24), (6, 22), (7, 21), (8, 29), (9, 24), (10, 27), (11, 17), (12, 7), (13, 7), (14, 4), (15, 3), (16, 1))

